In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge

# 1. 确认路径并读取第一步洗好的原始数据
drive_data_folder = '/content/drive/MyDrive/Colab Notebooks/DDA4210/DataSet' # 你的路径
file_path = os.path.join(drive_data_folder, 'NHANES_Cleaned_Step1.csv')
df = pd.read_csv(file_path)

# 去掉原始问卷列
phq9_cols = [f'PHQ9_{i}' for i in range(1, 10)]
df.drop(columns=phq9_cols, inplace=True, errors='ignore')

df_id = df['ID']
df_features = df.drop(columns=['ID'])
columns_names = df_features.columns

print("⏳ 开启大规模缺失值敏感性分析填补...")

# ==========================================
# 填补策略 A: 基于线性的 MICE (BayesianRidge)
# ==========================================
print("   -> 正在运行策略 A: Linear MICE...")
imputer_linear = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
data_linear = imputer_linear.fit_transform(df_features)
df_linear = pd.DataFrame(data_linear, columns=columns_names)
df_linear.insert(0, 'ID', df_id)

# ==========================================
# 填补策略 B: 基于随机森林的 MICE (MissForest 机制) - 极强非线性
# ==========================================
print("   -> 正在运行策略 B: Random Forest MICE (耗时稍长，请耐心等待)...")
# 为了控制运算时间，n_estimators 设为 50，max_iter 设为 5
rf_estimator = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
imputer_rf = IterativeImputer(estimator=rf_estimator, max_iter=5, random_state=42)
data_rf = imputer_rf.fit_transform(df_features)
df_rf = pd.DataFrame(data_rf, columns=columns_names)
df_rf.insert(0, 'ID', df_id)

# ==========================================
# 填补策略 C: 基础的中位数/众数填补 (Baseline)
# ==========================================
print("   -> 正在运行策略 C: Median Baseline...")
imputer_median = SimpleImputer(strategy='median')
data_median = imputer_median.fit_transform(df_features)
df_median = pd.DataFrame(data_median, columns=columns_names)
df_median.insert(0, 'ID', df_id)

# 对分类变量四舍五入恢复整数
int_cols = ['Gender', 'Education', 'Marital_Status', 'Cancer_History', 'Heart_Disease_History', 'Smoking_Status']
for df_temp in [df_linear, df_rf, df_median]:
    for col in int_cols:
        df_temp[col] = df_temp[col].round()

# ==========================================
# 分别保存三个数据集
# ==========================================
path_linear = os.path.join(drive_data_folder, 'NHANES_Imp_Linear.csv')
path_rf = os.path.join(drive_data_folder, 'NHANES_Imp_RF.csv')
path_median = os.path.join(drive_data_folder, 'NHANES_Imp_Median.csv')

df_linear.to_csv(path_linear, index=False)
df_rf.to_csv(path_rf, index=False)
df_median.to_csv(path_median, index=False)

print("三种填补策略全部完成并保存！")
print(f"文件列表:\n 1. {path_linear}\n 2. {path_rf}\n 3. {path_median}")

⏳ 开启大规模缺失值敏感性分析填补...
   -> 正在运行策略 A: Linear MICE...
   -> 正在运行策略 B: Random Forest MICE (耗时稍长，请耐心等待)...


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


   -> 正在运行策略 C: Median Baseline...
✅ 三种填补策略全部完成并保存！
📁 文件列表:
 1. /content/drive/MyDrive/Colab Notebooks/DDA4210/DataSet/NHANES_Imp_Linear.csv
 2. /content/drive/MyDrive/Colab Notebooks/DDA4210/DataSet/NHANES_Imp_RF.csv
 3. /content/drive/MyDrive/Colab Notebooks/DDA4210/DataSet/NHANES_Imp_Median.csv
